In [23]:
import os
import cv2
import yaml
import numpy as np
from IPython.display import display, clear_output
from PIL import Image
import ipynb.fs.full.detections as detections
from ipynb.fs.full.detections import run_detection_pipeline

**LOADING THE YAML FILE**

In [ ]:
with open("/content/config.yaml", "r") as file:
    config = yaml.safe_load(file)

In [13]:
dataset_path = config["project"]["base_raw_path"]

tracking_config = config["tracking"]
face_config = config["face_detection"]

TRACK_THRESH = tracking_config["track_thresh"]
MATCH_THRESH = tracking_config["match_thresh"]
TRACK_BUFFER = tracking_config["track_buffer"]
FRAME_RATE = tracking_config["frame_rate"]
CONF_THRESHOLD = face_config["conf_threshold"]
IOU_THRESHOLD = face_config["iou_threshold"]
MAX_DET = face_config["max_det"]
MIN_BOX_AREA=face_config["min_box_area"]
DEVICE = face_config["device"]

**HELPER FUNCTION**

In [14]:
def is_valid_frame(frame):
    return frame is not None and frame.size > 0

In [15]:
def track_faces(frame_id, image):
    results = detections.model.track(
        image,
        tracker="bytetrack.yaml",
        persist=True,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        max_det=MAX_DET,
        device=DEVICE,
        verbose=False
    )

    r = results[0]
    frame_tracks = []

    if r.boxes is None:
        return {"frame_id": frame_id, "tracks": []}

    boxes = r.boxes.xyxy.cpu().numpy()
    scores = r.boxes.conf.cpu().numpy()

    ids = None
    if r.boxes.id is not None:
        ids = r.boxes.id.cpu().numpy()

    for i, box in enumerate(boxes):

        x1, y1, x2, y2 = box
        score = scores[i]
        track_id = int(ids[i]) if ids is not None else -1

        frame_tracks.append({
            "track_id": track_id,
            "bbox": [int(x1), int(y1), int(x2), int(y2)],
            "confidence": float(score)
        })

    return {
        "frame_id": frame_id,
        "tracks": frame_tracks
    }


**MAIN FUNCTION**

In [16]:
def run_tracking_pipeline(dataset_path=None):

    if dataset_path is None:
        dataset_path = config["project"]["base_raw_path"]

    assert os.path.exists(dataset_path), f"Path not found: {dataset_path}"
    prepared_detections = run_detection_pipeline(dataset_path)
    
    assert detections.model is not None, "Model not loaded from detection"

    tracking_results = []

    frame_files = sorted(os.listdir(dataset_path))

    for frame_data in prepared_detections:

        frame_id = frame_data["frame_id"]

        frame = cv2.imread(os.path.join(dataset_path, frame_files[frame_id]))

        if not is_valid_frame(frame):
            continue

        tracking_results.append(track_faces(frame_id, frame))

    print(f"✅ Tracking done: {len(tracking_results)} frames")

    return tracking_results

In [24]:
if __name__ == "__main__":
    tracking_results=run_tracking_pipeline()

âœ… Model loaded: {0: 'FACE'}
âœ… Detection done: 2292 frames
âœ… Preparation done: 2292 frames ready for tracker
✅ Tracking done: 2292 frames
